## Normalización al Modelo del Cubo (Plata)

**Taller ETL – Cubo SECOP**
**Autor:** Jurani Zabala Hernandez

**Objetivo:** Tomar los datos crudos del área **bronce** y normalizarlos exactamente en las 7 dimensiones + 1 hecho definidos en `CuboDatos.ipynb`, aplicando limpieza, tipado y generación de llaves. El resultado se persiste en el área **silver (plata)**.

**Nota sobre `dim_tiempo` (dimensión de rol):** el hecho referencia la misma dimensión de tiempo 3 veces (`sk_tiempo_firma`, `sk_tiempo_inicio`, `sk_tiempo_fin`). Para esto, `dim_tiempo` se construye a partir de la **unión de las 3 fechas** (firma, inicio, fin) — cada fecha distinta que aparezca en cualquiera de los 3 roles genera una sola fila en `dim_tiempo`, y las 3 llaves foráneas del hecho usan la misma función de hash sobre la fecha correspondiente, garantizando que apunten correctamente a esa fila compartida.

El notebook es robusto a columnas faltantes (se rellenan como nulas y se reportan).

## 1. Sesión de Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

spark = (
    SparkSession.builder
    .appName("SECOP_Transformacion")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Sesión Spark inicializada:", spark.version)

26/08/24 01:56:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/24 01:56:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Sesión Spark inicializada: 3.1.2


## 2. Lectura del área Bronce

In [2]:
BRONZE_PATH = "hdfs://namenode:9000/datalake/bronze/secop/contratos"

df_bronce = spark.read.parquet(BRONZE_PATH)
print(f"Registros en bronce: {df_bronce.count()} | Columnas: {len(df_bronce.columns)}")
df_bronce.printSchema()

Registros en bronce: 20000 | Columnas: 88
root
 |-- nombre_entidad: string (nullable = true)
 |-- nit_entidad: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- localizaci_n: string (nullable = true)
 |-- orden: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- rama: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- proceso_de_compra: string (nullable = true)
 |-- id_contrato: string (nullable = true)
 |-- referencia_del_contrato: string (nullable = true)
 |-- estado_contrato: string (nullable = true)
 |-- codigo_de_categoria_principal: string (nullable = true)
 |-- descripcion_del_proceso: string (nullable = true)
 |-- tipo_de_contrato: string (nullable = true)
 |-- modalidad_de_contratacion: string (nullable = true)
 |-- justificacion_modalidad_de: string (nullable = true)
 |-- fecha_de_firma: string (nullable = true)
 |-- fecha_de_inicio_del_contrato: string (nullable = tr

## 3. Mapeo fuente - modelo del cubo

In [3]:
MAPA_COLUMNAS = {
    "id_contrato": "id_contrato",
    "codigo_entidad": "codigo_entidad",
    "nit_entidad": "nit_entidad",
    "nombre_entidad": "nombre_entidad",
    "orden": "orden",
    "sector": "sector",
    "rama": "rama",
    "entidad_centralizada": "centralizada",
    "codigo_proveedor": "codigo_proveedor",
    "tipodocproveedor": "tipo_documento",
    "documento_proveedor": "documento_proveedor",
    "proveedor_adjudicado": "nombre_proveedor",
    "es_grupo": "es_grupo",
    "es_pyme": "es_pyme",
    "fecha_de_firma": "fecha_firma",
    "fecha_de_inicio_del_contrato": "fecha_inicio",
    "fecha_de_fin_del_contrato": "fecha_fin",
    "modalidad_de_contratacion": "modalidad_contratacion",
    "justificacion_modalidad_de_contratacion": "justificacion_modalidad",
    "tipo_de_contrato": "tipo_contrato",
    "condiciones_de_entrega": "condiciones_entrega",
    "departamento": "departamento",
    "ciudad": "ciudad",
    "localizaci_n": "localizacion",
    "estado_contrato": "estado_contrato",
    "liquidaci_n": "liquidacion",
    "reversion": "reversion",
    "habilita_pago_adelantado": "habilita_pago_adelantado",
    "codigo_de_categoria_principal": "codigo_categoria_principal",
    "descripcion_del_proceso": "descripcion_proceso",
    "objeto_del_contrato": "objeto_contrato",
    "valor_del_contrato": "valor_contrato",
    "valor_facturado": "valor_facturado",
    "valor_pagado": "valor_pagado",
    "valor_pendiente_de_pago": "valor_pendiente_pago",
    "dias_adicionados": "dias_adicionados",
}

def safe_col(df, nombre_origen, alias):
    if nombre_origen in df.columns:
        return F.col(nombre_origen).alias(alias)
    else:
        return F.lit(None).cast("string").alias(alias)

faltantes = [c for c in MAPA_COLUMNAS if c not in df_bronce.columns]
print(f" Columnas del mapa no encontradas en bronce (se rellenan como NULL): {len(faltantes)}")
if faltantes:
    print(faltantes)

df_normalizado = df_bronce.select([
    safe_col(df_bronce, origen, destino) for origen, destino in MAPA_COLUMNAS.items()
])

df_normalizado.printSchema()

 Columnas del mapa no encontradas en bronce (se rellenan como NULL): 1
['justificacion_modalidad_de_contratacion']
root
 |-- id_contrato: string (nullable = true)
 |-- codigo_entidad: string (nullable = true)
 |-- nit_entidad: string (nullable = true)
 |-- nombre_entidad: string (nullable = true)
 |-- orden: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- rama: string (nullable = true)
 |-- centralizada: string (nullable = true)
 |-- codigo_proveedor: string (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- documento_proveedor: string (nullable = true)
 |-- nombre_proveedor: string (nullable = true)
 |-- es_grupo: string (nullable = true)
 |-- es_pyme: string (nullable = true)
 |-- fecha_firma: string (nullable = true)
 |-- fecha_inicio: string (nullable = true)
 |-- fecha_fin: string (nullable = true)
 |-- modalidad_contratacion: string (nullable = true)
 |-- justificacion_modalidad: string (nullable = true)
 |-- tipo_contrato: string (nullable =

## 4. Limpieza y tipado

Se convierten variables a sus formatos más adecuados, como valores, fechas y dias.

In [4]:
df_limpio = (
    df_normalizado
    .dropDuplicates(["id_contrato"])
    .na.drop(subset=["id_contrato"])
    .withColumn("fecha_firma", F.to_date("fecha_firma"))
    .withColumn("fecha_inicio", F.to_date("fecha_inicio"))
    .withColumn("fecha_fin", F.to_date("fecha_fin"))
    .withColumn("dias_adicionados", F.col("dias_adicionados").cast(IntegerType()))
    .withColumn("valor_contrato", F.col("valor_contrato").cast(DoubleType()))
    .withColumn("valor_facturado", F.col("valor_facturado").cast(DoubleType()))
    .withColumn("valor_pagado", F.col("valor_pagado").cast(DoubleType()))
    .withColumn("valor_pendiente_pago", F.col("valor_pendiente_pago").cast(DoubleType()))
    .filter(F.col("fecha_firma").isNotNull() & F.col("valor_contrato").isNotNull())
)

print(f"Registros después de limpieza: {df_limpio.count()}")
df_limpio.select("id_contrato", "estado_contrato", "valor_contrato", "fecha_firma").show(5, truncate=False)

Registros después de limpieza: 18581


+------------------+---------------+--------------+-----------+
|id_contrato       |estado_contrato|valor_contrato|fecha_firma|
+------------------+---------------+--------------+-----------+
|CO1.PCCNTR.1004753|Cerrado        |3.6391009979E8|2019-06-21 |
|CO1.PCCNTR.1334215|Cerrado        |4.1855E7      |2020-02-03 |
|CO1.PCCNTR.1358274|Modificado     |3.878586833E7 |2020-02-08 |
|CO1.PCCNTR.1415507|Aprobado       |5.836315E7    |2020-03-01 |
|CO1.PCCNTR.1444423|Aprobado       |1.08E7        |2020-03-13 |
+------------------+---------------+--------------+-----------+
only showing top 5 rows



## 5. Construcción de las 7 dimensiones

Cada dimensión se construye por sus valores únicos, con llave surrogate (`sk_*`) generada por `sha2` sobre su llave natural. `dim_tiempo` es especial: se construye a partir de la **unión** de las 3 columnas de fecha (firma, inicio, fin), ya que es una dimensión de rol compartida.

In [5]:
def construir_dimension(df, columnas_llave_natural, columnas_atributo, nombre_sk):
    columnas = list(dict.fromkeys(columnas_llave_natural + columnas_atributo))
    return (
        df.select(*columnas)
        .dropDuplicates(columnas_llave_natural)
        .withColumn(nombre_sk, F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("NA")) for c in columnas_llave_natural]), 256))
        .select(nombre_sk, *columnas)
    )

df_dim_entidad = construir_dimension(
    df_limpio, ["codigo_entidad"],
    ["nit_entidad", "nombre_entidad", "orden", "sector", "rama", "centralizada"],
    "sk_entidad",
)

df_dim_proveedor = construir_dimension(
    df_limpio, ["documento_proveedor"],
    ["codigo_proveedor", "tipo_documento", "nombre_proveedor", "es_grupo", "es_pyme"],
    "sk_proveedor",
)

df_dim_modalidad = construir_dimension(
    df_limpio, ["modalidad_contratacion"],
    ["justificacion_modalidad", "tipo_contrato", "condiciones_entrega"],
    "sk_modalidad",
)

df_dim_ubicacion = construir_dimension(
    df_limpio, ["departamento", "ciudad", "localizacion"], [], "sk_ubicacion"
)

df_dim_estado = construir_dimension(
    df_limpio, ["estado_contrato"],
    ["liquidacion", "reversion", "habilita_pago_adelantado"],
    "sk_estado",
)

df_dim_categoria = construir_dimension(
    df_limpio, ["codigo_categoria_principal"],
    ["descripcion_proceso", "objeto_contrato"],
    "sk_categoria",
)

# --- dim_tiempo: dimensión de rol, construida como UNIÓN de las 3 fechas ---
fechas_union = (
    df_limpio.select(F.col("fecha_firma").alias("fecha"))
    .unionByName(df_limpio.select(F.col("fecha_inicio").alias("fecha")))
    .unionByName(df_limpio.select(F.col("fecha_fin").alias("fecha")))
    .filter(F.col("fecha").isNotNull())
    .dropDuplicates(["fecha"])
)

df_dim_tiempo = (
    fechas_union
    .withColumn("sk_tiempo", F.sha2(F.col("fecha").cast("string"), 256))
    .withColumn("anio", F.year("fecha"))
    .withColumn("semestre", F.when(F.month("fecha") <= 6, 1).otherwise(2))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("nombre_mes", F.date_format("fecha", "MMMM"))
    .withColumn("dia", F.dayofmonth("fecha"))
    .select("sk_tiempo", "fecha", "anio", "semestre", "trimestre", "mes", "nombre_mes", "dia")
)

for nombre, df in [
    ("dim_entidad", df_dim_entidad), ("dim_proveedor", df_dim_proveedor),
    ("dim_tiempo", df_dim_tiempo), ("dim_modalidad", df_dim_modalidad),
    ("dim_ubicacion", df_dim_ubicacion), ("dim_estado_contrato", df_dim_estado),
    ("dim_categoria", df_dim_categoria),
]:
    print(f"{nombre}: {df.count()} registros")

dim_entidad: 2005 registros


dim_proveedor: 18001 registros


dim_tiempo: 3255 registros


dim_modalidad: 14 registros


dim_ubicacion: 611 registros


dim_estado_contrato: 9 registros


dim_categoria: 1471 registros


## 6. Construcción de la tabla de hechos

Las 3 llaves de tiempo (`sk_tiempo_firma`, `sk_tiempo_inicio`, `sk_tiempo_fin`) se calculan con el **mismo hash** usado en `dim_tiempo`, garantizando que apunten a las filas correctas de esa dimensión compartida. `cantidad_contratos` se fija en 1 (grano del hecho = un contrato por fila; queda como métrica que suma para agregaciones).

In [6]:
df_hechos = (
    df_limpio
    .withColumn("sk_entidad", F.sha2(F.coalesce(F.col("codigo_entidad"), F.lit("NA")), 256))
    .withColumn("sk_proveedor", F.sha2(F.coalesce(F.col("documento_proveedor"), F.lit("NA")), 256))
    .withColumn("sk_tiempo_firma", F.sha2(F.col("fecha_firma").cast("string"), 256))
    .withColumn("sk_tiempo_inicio", F.when(F.col("fecha_inicio").isNotNull(), F.sha2(F.col("fecha_inicio").cast("string"), 256)))
    .withColumn("sk_tiempo_fin", F.when(F.col("fecha_fin").isNotNull(), F.sha2(F.col("fecha_fin").cast("string"), 256)))
    .withColumn("sk_modalidad", F.sha2(F.coalesce(F.col("modalidad_contratacion"), F.lit("NA")), 256))
    .withColumn("sk_ubicacion", F.sha2(F.concat_ws("|", F.coalesce(F.col("departamento"), F.lit("NA")), F.coalesce(F.col("ciudad"), F.lit("NA")), F.coalesce(F.col("localizacion"), F.lit("NA"))), 256))
    .withColumn("sk_estado", F.sha2(F.coalesce(F.col("estado_contrato"), F.lit("NA")), 256))
    .withColumn("sk_categoria", F.sha2(F.coalesce(F.col("codigo_categoria_principal"), F.lit("NA")), 256))
    .withColumn("cantidad_contratos", F.lit(1).cast(IntegerType()))
    .withColumn("anio_firma", F.year("fecha_firma"))
    .select(
        "id_contrato", "sk_entidad", "sk_proveedor", "sk_tiempo_firma", "sk_tiempo_inicio", "sk_tiempo_fin",
        "sk_modalidad", "sk_ubicacion", "sk_estado", "sk_categoria",
        "valor_contrato", "valor_facturado", "valor_pagado", "valor_pendiente_pago",
        "dias_adicionados", "cantidad_contratos", "anio_firma",
    )
)

print(f"hecho_contratos: {df_hechos.count()} registros")
df_hechos.select("id_contrato", "valor_contrato", "sk_entidad", "anio_firma").show(5, truncate=False)

hecho_contratos: 18581 registros


+------------------+--------------+----------------------------------------------------------------+----------+
|id_contrato       |valor_contrato|sk_entidad                                                      |anio_firma|
+------------------+--------------+----------------------------------------------------------------+----------+
|CO1.PCCNTR.1004753|3.6391009979E8|d58d94c553375acc2c155d57f65a4107abdec5d50c64b6db1166a9383604eb9c|2019      |
|CO1.PCCNTR.1334215|4.1855E7      |1e881e1e48f20343c00d6b623900702921aa4ec9c604140832e2f40ecf6d4134|2020      |
|CO1.PCCNTR.1358274|3.878586833E7 |d01134138abbc5971d13fe8ccd5b8c6409653a4efe1668f9aa4c4d16d54d1df6|2020      |
|CO1.PCCNTR.1415507|5.836315E7    |b3cd809a27095a60e91786beb8a0c17a2ca701804d3504e7164f375bf88bcc51|2020      |
|CO1.PCCNTR.1444423|1.08E7        |db332d69146a50fa907a08da6586e14348f3873eacf844f04a5cab4f5e06e058|2020      |
+------------------+--------------+----------------------------------------------------------------+----

## 7. Persistencia en el área Plata

In [7]:
SILVER_BASE = "hdfs://namenode:9000/datalake/silver/secop"

destinos = {
    "hecho_contratos": df_hechos,
    "dim_entidad": df_dim_entidad,
    "dim_proveedor": df_dim_proveedor,
    "dim_tiempo": df_dim_tiempo,
    "dim_modalidad": df_dim_modalidad,
    "dim_ubicacion": df_dim_ubicacion,
    "dim_estado_contrato": df_dim_estado,
    "dim_categoria": df_dim_categoria,
}

for nombre, df in destinos.items():
    ruta = f"{SILVER_BASE}/{nombre}"
    df.coalesce(4).write.mode("overwrite").parquet(ruta)
    print(f"✅ {nombre} -> {ruta}")

26/08/24 01:58:23 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✅ hecho_contratos -> hdfs://namenode:9000/datalake/silver/secop/hecho_contratos


✅ dim_entidad -> hdfs://namenode:9000/datalake/silver/secop/dim_entidad


✅ dim_proveedor -> hdfs://namenode:9000/datalake/silver/secop/dim_proveedor


✅ dim_tiempo -> hdfs://namenode:9000/datalake/silver/secop/dim_tiempo


✅ dim_modalidad -> hdfs://namenode:9000/datalake/silver/secop/dim_modalidad


✅ dim_ubicacion -> hdfs://namenode:9000/datalake/silver/secop/dim_ubicacion


✅ dim_estado_contrato -> hdfs://namenode:9000/datalake/silver/secop/dim_estado_contrato


✅ dim_categoria -> hdfs://namenode:9000/datalake/silver/secop/dim_categoria


## Conclusiones

- Se normalizaron los datos crudos del área bronce al modelo exacto del cubo propuesto para este taller: 7 dimensiones + 1 hecho.
- `dim_tiempo` se implementó correctamente como **dimensión de rol**, construida por unión de las 3 fechas del contrato, con las 3 llaves foráneas del hecho apuntando consistentemente a ella mediante el mismo hash.
- Los resultados quedaron persistidos en el área **plata**, listos para `Cargue.ipynb`.